# Caso 2. Viento: v0.2.0 frente a la propuesta v0.2.1 (cola en 12 m/s y escala Saffir-Simpson), en malla y por municipio

**Qué se compara.** La lluvia es la de v0.2.0 en los tres experimentos; cambia solo el viento, en dos pasos:

| Experimento | Cola de viento | Escala de H_W | Configuración |
|---|---|---|---|
| v0.2.0 | Hasta R_W06 (2 m/s) | Relativa a la Vmax de la tormenta | `v021/v020.yaml` |
| corte 12 m/s | Hasta donde V = 12 m/s (23.3 kt) | Relativa a Vmax | `v021/w12.yaml` |
| v0.2.1 Saffir-Simpson | Hasta 12 m/s | **Absoluta**, homologada con Saffir-Simpson | `v021/ss.yaml` |

**La propuesta de homologación** (notebook 11, A4): cola de 12 m/s a 34 kt → clase 1; tormenta tropical (34–63 kt) → clase 2; huracán 1–2 (64–95 kt) → clase 3; huracán 3 (96–112 kt) → clase 4; huracán 4–5 (≥ 113 kt) → clase 5. Así la clase de viento de un píxel dice qué viento hubo ahí, igual en todas las tormentas. En v0.2.0 dice qué tan cerca estaba el píxel del máximo de su propia tormenta: una tormenta tropical de 45 kt llega a la clase 5.

**Qué se espera antes de ver los mapas:**
1. El corte en 12 m/s solo reduce: Δclase ≤ 0, y los cambios están lejos del centro (la clase 1 de la cola).
2. Saffir-Simpson baja las clases altas de las tormentas débiles y de la periferia de los huracanes: los casos menos intensos sobre México (Cristóbal 2020 tocó tierra en Campeche como tormenta tropical) deben bajar más que Patricia 2015 y Odile 2014. La sección 1 da la intensidad de cada caso.
3. Donde la lluvia ya era de clase alta, ITCHI no cambia (la lluvia domina la unión).
4. En el ojo, la propuesta asigna el peligro de Vmax (v0.2.0 asignaba 0.2); en el máximo del evento esto casi no se nota, porque la pared del ojo pasa por esos píxeles.

**Limitaciones del viento** (para la discusión, plan v0.3 decisión 6). El viento es el sostenido de 1 min del perfil W06 (PA21) calibrado a R34, como en IBTrACS; Saffir-Simpson se define para ese mismo viento sobre terreno abierto. Sobre tierra no se reduce por rugosidad ni por el decaimiento tierra adentro, así que el viento de los píxeles del interior está sobrestimado. El corte en 12 m/s (Beaufort 6) todavía no tiene una justificación física de daño; si el argumento es de daño, habría que pasar a ráfaga.

**Entradas.** Las mismas del notebook 12 (`python casos_v021.py`).

## 0. Configuración y datos

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from itchi.config import get_project_root

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
import casos_v021 as cv       # noqa: E402
import figuras_v021 as F      # noqa: E402

CFG = cv.ConfigV021()          # rutas (D:/MSWEP_V316_6H, climatologías, IBTrACS, ROCLOUD, municipios)
FORZAR = False                 # True: recalcula aunque exista el caché de outputs/diagnostics/avance_v021/
MODO_MUNICIPIO = "poligonos"   # "poligonos" (shapefile) o "malla" (cada píxel con el valor de su municipio)
FIG = CFG.fig_dir / "caso2_viento"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()


assert CFG.mswep_6h_dir.is_dir(), f"no existe {CFG.mswep_6h_dir}"
for d, f in CFG.clim.items():
    assert f.exists(), f"falta {f.name}"

In [ ]:
CASOS = list(cv.CASOS_1_2)                 # Emily 2005, Alex 2010, Patricia 2015, Cristóbal 2020, Odile 2014
DATA = {c: cv.load_or_compute(c, CFG, force=FORZAR) for c in CASOS}
DECL = cv.load_declaratorias(CFG)
for c, C in DATA.items():
    ds = C["ds"]
    print(f"{c}: {ds.sizes['time']} posiciones que alcanzan México "
          f"({pd.Timestamp(ds.time.values[0]):%Y-%m-%d %H} a {pd.Timestamp(ds.time.values[-1]):%Y-%m-%d %H} UTC), "
          f"Vmax máx {float(ds.track_vmax_kt.max()):.0f} kt")

In [ ]:
EXP = ["v020", "w12", "ss"]
REF = "v020"

## 1. Intensidad de cada caso sobre México

Vmax máxima en las posiciones usadas y categoría Saffir-Simpson. Es la intensidad que limita la clase de viento con la propuesta.

In [ ]:
from itchi.constants import SAFFIR_SIMPSON_KT


def categoria(v):
    # Categoría Saffir-Simpson de un viento sostenido de 1 min (kt)
    cat = "DT"
    for k, lim in SAFFIR_SIMPSON_KT.items():
        if v >= lim:
            cat = "TT" if k == "TS" else k
    return cat


rows = []
for c, C in DATA.items():
    ds = C["ds"]
    v = float(ds.track_vmax_kt.max())
    hw = {cv.label(e): int(cv.siat_class(ds["H_W"].sel(exp=e).max().values)) for e in EXP}
    rows.append(dict(caso=c, vmax_max_kt=v, categoria=categoria(v), **{f"clase H_W máx · {k}": x for k, x in hw.items()}))
pd.DataFrame(rows).set_index("caso")

## 2. Peligro por viento solo (H_W_max)

Máximo del evento de H_W en malla para los tres experimentos, con contornos en los límites de clase.

**Cómo leerla.**
- **v0.2.0 → corte 12 m/s.** El área coloreada se encoge: la cola termina más cerca del centro. Dentro de R34 no cambia nada.
- **corte 12 m/s → Saffir-Simpson.** El área es la misma; cambia el valor. En las tormentas tropicales el máximo baja a 0.2–0.4; en los huracanes intensos (Patricia, Odile) el núcleo sigue cerca de 1.

In [ ]:
for c, C in DATA.items():
    guardar(F.fig_mapa_campo(C, "H_W", EXP, cmap="OrRd"), f"{c.lower()}_HW_v020_w12_ss")

## 3. Mapas del evento: malla y municipio

Para cada caso, una figura de 2 × 4: clase de ITCHI_max con v0.2.0, con el corte en 12 m/s y con la propuesta Saffir-Simpson, y Δclase (propuesta − v0.2.0); arriba en malla y abajo por municipio.

**Cómo leerla.**
- **Δclase negativo lejos del centro.** Es el corte en 12 m/s: la cola de v0.2.0 ponía clase 1 en todo el círculo de R_W06.
- **Δclase negativo cerca de la trayectoria.** Es Saffir-Simpson: el viento de una tormenta débil ya no llega a las clases 4–5.
- **Δclase = 0 sobre los núcleos de lluvia.** Expectativa 3: ahí manda la lluvia.
- **Δclase positivo.** Solo puede aparecer en el ojo (peligro de Vmax en lugar de 0.2) o donde el viento local supera el máximo de su clase relativa. Si aparece fuera del centro de un huracán intenso, revisarlo.

In [ ]:
for c, C in DATA.items():
    fig = F.fig_mapa_evento(C, EXP, ref=REF, mode=MODO_MUNICIPIO,
                            title=f"{c}: clase de ITCHI_max, v0.2.0 frente al corte en 12 m/s y a Saffir-Simpson")
    guardar(fig, f"{c.lower()}_evento_v020_w12_ss")

## 4. Una posición: el pico del evento sobre México

Campo instantáneo en la posición con mayor ITCHI acumulado sobre tierra, con los radios: R_direct (negro), R_W06 (naranja), atribución (magenta). Aquí se ve la cola completa de v0.2.0 hasta R_W06 y su fin en 12 m/s.

In [ ]:
for c, C in DATA.items():
    t = cv.peak_time(C, "v020")
    guardar(F.fig_mapa_posicion(C, t, EXP, mode=MODO_MUNICIPIO), f"{c.lower()}_posicion_{t:%Y%m%d%H}_v020_w12_ss")

## 5. Conteos por clase y prueba ordinal

**Cómo leerla.** El corte en 12 m/s vacía sobre todo la clase 1; Saffir-Simpson mueve municipios de las clases 4–5 a las 2–3 en las tormentas débiles. En la prueba ordinal, P(D) debe crecer con la clase en las tres versiones. Si la propuesta mejora el orden (por ejemplo, P(D | clase 1) baja porque la clase 1 ya no cubre todo R_W06), es evidencia a favor; si lo rompe, en contra.

In [ ]:
TABLAS = {c: cv.class_counts(C, EXP) for c, C in DATA.items()}
guardar(F.fig_conteos(TABLAS, [cv.label(e) for e in EXP], "municipios"), "conteos_municipios_v020_w12_ss")
display(pd.concat(TABLAS, names=["caso"]))

ORD = []
for c, C in DATA.items():
    inicio = cv.declaratoria_inicio(DECL, cv.CASES[c]["atcf"])
    t = cv.tabla_ordinal(C, set(inicio.index), EXP)
    t.insert(0, "caso", c)
    ORD.append(t)
ORD = pd.concat(ORD, ignore_index=True)
TOTAL = ORD.groupby(["exp", "clase"])[["municipios", "con_declaratoria"]].sum()
TOTAL["P_D"] = TOTAL["con_declaratoria"] / TOTAL["municipios"]
display(TOTAL.unstack("exp").round(3))
ORD.to_csv(FIG / "prueba_ordinal_v020_w12_ss.csv", index=False)

## 6. Para la discusión

1. **¿El corte en 12 m/s elimina la clase 1 «de fondo» sin tocar el núcleo?** Secciones 2 y 3.
2. **¿Saffir-Simpson ordena los casos por intensidad?** Sección 1 contra la sección 3: con la propuesta, la clase de viento máxima de cada caso debe seguir su categoría (Patricia y Odile arriba, Cristóbal abajo). Con v0.2.0 todos llegan a la clase 5.
3. **¿Qué agrupación de categorías?** La propuesta junta H1–H2 y H4–H5. La alternativa del plan v0.3 (4 niveles: < 34, TT, H1–H2, ≥ 96 kt) se prueba cambiando `wind.saffir_simpson_nodes`, por ejemplo `[[34, 0.25], [64, 0.5], [96, 0.75], [113, 1.0]]`.
4. **¿Viento sobre tierra?** Sin reducción por rugosidad, el peligro por viento tierra adentro está sobrestimado (decisión 6 del plan v0.3).